# DASHBOARD INTERACTIVO SERIE A TIM 2025/26
**Análisis avanzado con ipywidgets + escudos**


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

# Cargar datos limpios
BASE_PATH = Path("data/cleaned")
df_matches = pd.read_csv(BASE_PATH / "matches_clean.csv")
df_players = pd.read_csv(BASE_PATH / "player_stats_clean.csv")
df_shots = pd.read_csv(BASE_PATH / "shot_map_clean.csv")
df_teams = pd.read_csv(BASE_PATH / "team_stats_clean.csv")

print("✅ Datos cargados:")
print(f"Matches: {len(df_matches)} partidos")
print(f"Jugadores: {len(df_players)} registros")
print(f"Tiros: {len(df_shots)} disparos")


✅ Datos cargados:
Matches: 300 partidos
Jugadores: 14029 registros
Tiros: 7458 disparos


## RANKING GOLEADORES + EQUIPOS (Interactivo)


In [ ]:
# Widgets
equipos = sorted(df_players['team'].unique())
dropdown_equipo = widgets.Dropdown(options=['TODOS'] + equipos, 
                                  value='TODOS', 
                                  description='Equipo:')

min_goles = widgets.IntSlider(min=0, max=20, value=0, description='Min Goles')

def plot_goleadores(equipo, min_goles):
    if equipo == 'TODOS':
        df_plot = df_players
    else:
        df_plot = df_players[df_players['team'] == equipo]
    
    top_scorers = df_plot.groupby(['player', 'team'])['goals'].sum().reset_index()
    top_scorers = top_scorers[top_scorers['goals'] >= min_goles]
    top_scorers = top_scorers.nlargest(10, 'goals')
    
    fig = px.bar(top_scorers, x='player', y='goals', 
                color='team',
                title=f"TOP 10 Goleadores (Min {min_goles} goles)")
    fig.show()

widgets.interact(plot_goleadores, equipo=dropdown_equipo, min_goles=min_goles)


interactive(children=(Dropdown(description='Equipo:', options=('TODOS', 'Atalanta', 'Bologna', 'Cagliari', 'Co…

<function __main__.plot_goleadores(equipo, min_goles)>

## TABLA POSICIONES (Puntos + xG)


In [7]:
# Calcular tabla posiciones REAL
results = []
for _, match in df_matches.iterrows():
    home = match['home_team']
    away = match['away_team']
    home_score = match['home_score']
    away_score = match['away_score']
    
    if home_score > away_score:
        home_points = 3
        away_points = 0
    elif home_score == away_score:
        home_points = 1
        away_points = 1
    else:
        home_points = 0
        away_points = 3
    
    results.append({'team': home, 'gf': home_score, 'ga': away_score, 'points': home_points})
    results.append({'team': away, 'gf': away_score, 'ga': home_score, 'points': away_points})

df_results = pd.DataFrame(results)
tabla = df_results.groupby('team').agg({
    'gf': 'sum',
    'ga': 'sum', 
    'points': 'sum',
    'team': 'count'  # PJ
}).rename(columns={'team': 'pj'}).reset_index()

tabla['gd'] = tabla['gf'] - tabla['ga']
tabla = tabla.sort_values(['points', 'gd', 'gf'], ascending=[False, False, False]).reset_index(drop=True)
tabla['pos'] = range(1, len(tabla) + 1)

print("Tabla de posiciones correcta:")
print(tabla.head(6))

# Gráfico
fig = px.bar(tabla.head(6), x='points', y='team', 
             color='team', orientation='h',
             title="TABLA TOP 6 - Serie A TIM (Puntos)")
fig.update_xaxes(title="Puntos")
fig.update_yaxes(title="Equipo")
fig.show()

# Tabla coloreada
tabla.head(6).style.background_gradient(cmap='viridis', subset=['points'])



Tabla de posiciones correcta:
       team  gf  ga  points  pj  gd  pos
0     Inter  66  24      69  30  42    1
1     Milan  47  23      63  30  24    2
2    Napoli  46  30      62  30  16    3
3      Como  53  22      57  30  31    4
4  Juventus  52  29      54  30  23    5
5      Roma  40  23      54  30  17    6


,team,gf,ga,points,pj,gd,pos
0,Inter,66,24,69,30,42,1
1,Milan,47,23,63,30,24,2
2,Napoli,46,30,62,30,16,3
3,Como,53,22,57,30,31,4
4,Juventus,52,29,54,30,23,5
5,Roma,40,23,54,30,17,6


## ANÁLISIS GENERAL DE LA LIGA SERIE A TIM 2025/26

En esta sección realizaremos un análisis general de la temporada actual de la Serie A italiana, incluyendo estadísticas globales, tendencias y comparaciones.

In [14]:
# Estadísticas generales de la liga
total_matches = len(df_matches)
total_goals = df_matches['home_score'].sum() + df_matches['away_score'].sum()
avg_goals_per_match = total_goals / total_matches
max_goals_in_match = (df_matches['home_score'] + df_matches['away_score']).max()

# Resultados de partidos
home_wins = (df_matches['home_score'] > df_matches['away_score']).sum()
away_wins = (df_matches['home_score'] < df_matches['away_score']).sum()
draws = (df_matches['home_score'] == df_matches['away_score']).sum()

# Equipos más goleadores
team_goals = df_matches.groupby('home_team')['home_score'].sum().add(
    df_matches.groupby('away_team')['away_score'].sum(), fill_value=0
).sort_values(ascending=False)

print("📊 ANÁLISIS GENERAL DE LA SERIE A TIM 2025/26")
print(f"Total de partidos jugados: {total_matches}")
print(f"Total de goles marcados: {total_goals}")
print(f"Promedio de goles por partido: {avg_goals_per_match:.2f}")
print(f"Máximo de goles en un partido: {max_goals_in_match}")
print(f"Victorias locales: {home_wins} ({home_wins/total_matches*100:.1f}%)")
print(f"Victorias visitantes: {away_wins} ({away_wins/total_matches*100:.1f}%)")
print(f"Empates: {draws} ({draws/total_matches*100:.1f}%)")
print("\n🏆 EQUIPOS MÁS GOLEADORES:")
print(team_goals.head(5))

📊 ANÁLISIS GENERAL DE LA SERIE A TIM 2025/26
Total de partidos jugados: 300
Total de goles marcados: 733
Promedio de goles por partido: 2.44
Máximo de goles en un partido: 8
Victorias locales: 119 (39.7%)
Victorias visitantes: 102 (34.0%)
Empates: 79 (26.3%)

🏆 EQUIPOS MÁS GOLEADORES:
home_team
Inter       66
Como        53
Juventus    52
Milan       47
Napoli      46
dtype: int64


In [15]:
# Visualización: Distribución de resultados
labels = ['Victorias Local', 'Victorias Visitante', 'Empates']
values = [home_wins, away_wins, draws]
colors = ['#FF9999', '#66B3FF', '#99FF99']

fig_results = go.Figure(data=[go.Pie(labels=labels, values=values, marker_colors=colors)])
fig_results.update_layout(title="Distribución de Resultados en la Liga")
fig_results.show()

# Visualización: Goles por equipo (Top 10)
top_teams_goals = team_goals.head(10)
fig_goals = px.bar(top_teams_goals, x=top_teams_goals.index, y=top_teams_goals.values,
                   title="Top 10 Equipos Más Goleadores",
                   labels={'x': 'Equipo', 'y': 'Goles'})
fig_goals.show()

# Estadísticas adicionales: Goles por jornada
goals_per_round = df_matches.groupby('round')['home_score'].sum() + df_matches.groupby('round')['away_score'].sum()
fig_rounds = px.line(goals_per_round, title="Goles por Jornada",
                     labels={'index': 'Jornada', 'value': 'Goles Totales'})
fig_rounds.show()

In [17]:
# Análisis adicional: Estadísticas de equipos
# Promedio de posesión, tiros, etc.
df_teams['total_shots'] = df_teams['totalShotsInsideBox'] + df_teams['totalShotsOutsideBox']
team_stats_avg = df_teams.groupby('team').agg({
    'ballPossession': 'mean',
    'total_shots': 'mean',
    'totalShotsOnGoal': 'mean',
    'cornerKicks': 'mean'
}).round(2)

print("\n📈 ESTADÍSTICAS PROMEDIO POR EQUIPO:")
print(team_stats_avg.head())

# Correlación entre tiros y goles
shots_goals = df_teams.groupby('team').agg({
    'total_shots': 'sum',
    'totalShotsOnGoal': 'sum'
}).join(team_goals.rename('goals'))

correlation = shots_goals.corr()
print("\n🔗 CORRELACIÓN ENTRE TIROS Y GOLES:")
print(correlation)

# Visualización: Scatter plot tiros vs goles
fig_scatter = px.scatter(shots_goals, x='total_shots', y='goals', text=shots_goals.index,
                         title="Tiros Totales vs Goles Marcados",
                         labels={'total_shots': 'Tiros Totales', 'goals': 'Goles'})
fig_scatter.update_traces(textposition='top center')
fig_scatter.show()


📈 ESTADÍSTICAS PROMEDIO POR EQUIPO:
           ballPossession  total_shots  totalShotsOnGoal  cornerKicks
team                                                                 
Atalanta            55.13        14.77             14.77         5.73
Bologna             54.90        13.43             13.43         4.53
Cagliari            46.00        10.00             10.00         3.43
Como                61.73        14.27             14.27         4.23
Cremonese           45.67         9.17              9.17         3.27

🔗 CORRELACIÓN ENTRE TIROS Y GOLES:
                  total_shots  totalShotsOnGoal     goals
total_shots          1.000000          1.000000  0.882931
totalShotsOnGoal     1.000000          1.000000  0.882931
goals                0.882931          0.882931  1.000000
